In [1]:
!pip install pandas numpy scikit-learn lightgbm xgboost shap lime kagglehub imbalanced-learn joblib streamlit matplotlib seaborn

     ---------------------------------------- 0.0/275.7 kB ? eta -:--:--
     ---------------------------------------- 0.0/275.7 kB ? eta -:--:--
     ---------------------------------------- 0.0/275.7 kB ? eta -:--:--
     ---------------------------------------- 0.0/275.7 kB ? eta -:--:--
     - -------------------------------------- 10.2/275.7 kB ? eta -:--:--
     - -------------------------------------- 10.2/275.7 kB ? eta -:--:--
     - -------------------------------------- 10.2/275.7 kB ? eta -:--:--
     - -------------------------------------- 10.2/275.7 kB ? eta -:--:--
     - -------------------------------------- 10.2/275.7 kB ? eta -:--:--
     - -------------------------------------- 10.2/275.7 kB ? eta -:--:--
     ---- --------------------------------- 30.7/275.7 kB 93.5 kB/s eta 0:00:03
     ---- --------------------------------- 30.7/275.7 kB 93.5 kB/s eta 0:00:03
     ---- --------------------------------- 30.7/275.7 kB 93.5 kB/s eta 0:00:03
     ----- -------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.0.2 which is incompatible.
gradio 4.39.0 requires httpx>=0.24.1, but you have httpx 0.13.3 which is incompatible.
tensorboard 2.17.0 requires protobuf!=4.24.0,<5.0.0,>=3.19.6, but you have protobuf 5.28.2 which is incompatible.
tensorflow-intel 2.17.0 requires ml-dtypes<0.5.0,>=0.3.1, but you have ml-dtypes 0.5.4 which is incompatible.
tensorflow-intel 2.17.0 requires numpy<2.0.0,>=1.23.5; python_version <= "3.11", but you have numpy 2.0.2 which is incompatible.
tensorflow-intel 2.17.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 5.28.2 which is incompatible.
thinc 8.2.5 requires numpy<2.0.0,>=1.19.0; python_version >= "3.9", but you have numpy 2.0.2 which is incompatible.

[notice] A 

In [1]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import lime
import lime.lime_tabular
import numpy as np
import pandas as pd
import gradio as gr
from sklearn.base import BaseEstimator, TransformerMixin


class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        epsilon = 1e-6

        if "Amount" in X_copy.columns and "Income" in X_copy.columns:
            X_copy["Debt_to_Income_Ratio"] = X_copy["Amount"] / (X_copy["Income"] + epsilon)
        if "Age" in X_copy.columns and "Emp_length" in X_copy.columns:
            X_copy["Age_x_EmpLength"] = X_copy["Age"] * X_copy["Emp_length"]
        if "Amount" in X_copy.columns and "Cred_length" in X_copy.columns:
            X_copy["Amount_per_CredLength"] = X_copy["Amount"] / (X_copy["Cred_length"] + epsilon)
        if "Rate" in X_copy.columns and "Amount" in X_copy.columns:
            X_copy["Rate_per_Amount"] = X_copy["Rate"] / (X_copy["Amount"] + epsilon)
        if "Income" in X_copy.columns and "Age" in X_copy.columns:
            X_copy["Income_per_Age"] = X_copy["Income"] / (X_copy["Age"] + epsilon)

        return X_copy


def _to_dataframe(X_array, columns):
    return pd.DataFrame(X_array, columns=columns)


best_lgb_model = joblib.load("best_lgbm_model.joblib")
full_ml_pipeline = joblib.load("full_ml_pipeline.joblib")
model_feature_names = joblib.load("model_feature_names.joblib")
lime_background_data = joblib.load("lime_background_data.joblib")

low_risk_threshold = 0.20
moderate_risk_threshold = 0.50


def categorize_risk(probability):
    if probability < low_risk_threshold:
        return "Low Risk"
    elif low_risk_threshold <= probability < moderate_risk_threshold:
        return "Moderate Risk"
    return "High Risk"


def map_to_decision(risk_category):
    if risk_category == "Low Risk":
        return "Approve"
    elif risk_category == "Moderate Risk":
        return "Review"
    return "Reject"


def calculate_interest_rate(risk_category):
    if risk_category == "Low Risk":
        return "6.0%"
    elif risk_category == "Moderate Risk":
        return "10.0%"
    return "15.0%"


def get_plain_english_explanation(feature, weight, raw_input_data):
    parts = feature.split(" ")
    feature_name = parts[0]
    impact_direction = "increased" if weight > 0 else "decreased"
    impact_strength = "strongly" if abs(weight) > 0.15 else "moderately" if abs(weight) > 0.05 else "slightly"

    if feature_name.startswith("Home_"):
        original_home_value = raw_input_data.get("Home", "N/A")
        if feature_name == f"Home_{original_home_value}":
            return f"Being a {original_home_value} owner {impact_strength} {impact_direction} the risk."
        return f"Not being a {feature_name.replace('Home_', '')} owner {impact_strength} {impact_direction} the risk."

    if feature_name.startswith("Intent_"):
        original_intent_value = raw_input_data.get("Intent", "N/A")
        if feature_name == f"Intent_{original_intent_value}":
            return f"The loan intent for {original_intent_value} {impact_strength} {impact_direction} the risk."
        return f"Not having a loan intent for {feature_name.replace('Intent_', '')} {impact_strength} {impact_direction} the risk."

    if feature_name == "Debt_to_Income_Ratio":
        return f"The Debt-to-Income Ratio ({raw_input_data.get('Amount', 0) / (raw_input_data.get('Income', 1e-6)):.2f}) {impact_strength} {impact_direction} the risk."
    if feature_name == "Percent_income":
        return f"The Loan Amount as a Percentage of Income ({raw_input_data.get('Percent_income', 0):.2f}) {impact_strength} {impact_direction} the risk."
    if feature_name == "Age_x_EmpLength":
        return f"The combination of Age and Employment Length {impact_strength} {impact_direction} the risk."
    if feature_name == "Amount_per_CredLength":
        return f"The Loan Amount relative to Credit History Length {impact_strength} {impact_direction} the risk."
    if feature_name == "Rate_per_Amount":
        return f"The Interest Rate relative to Loan Amount {impact_strength} {impact_direction} the risk."
    if feature_name == "Age":
        return f"The applicant's Age ({raw_input_data.get('Age', 'N/A')} years) {impact_strength} {impact_direction} the risk."
    if feature_name == "Income":
        return f"The applicant's Annual Income (${raw_input_data.get('Income', 0):,.0f}) {impact_strength} {impact_direction} the risk."
    if feature_name == "Emp_length":
        return f"The Employment Length ({raw_input_data.get('Emp_length', 'N/A')} years) {impact_strength} {impact_direction} the risk."
    if feature_name == "Amount":
        return f"The Loan Amount (${raw_input_data.get('Amount', 0):,.0f}) {impact_strength} {impact_direction} the risk."
    if feature_name == "Rate":
        return f"The Interest Rate ({raw_input_data.get('Rate', 'N/A')}%) {impact_strength} {impact_direction} the risk."
    if feature_name == "Cred_length":
        return f"The Credit History Length ({raw_input_data.get('Cred_length', 'N/A')} years) {impact_strength} {impact_direction} the risk."

    return f"A factor related to {feature_name} {impact_strength} {impact_direction} the risk."


def assess_credit_risk(age, income, emp_length, amount, rate, cred_length, home_ownership, loan_intent):
    try:
        raw_input_data = {
            "Age": age,
            "Income": income,
            "Home": home_ownership,
            "Emp_length": emp_length,
            "Intent": loan_intent,
            "Amount": amount,
            "Rate": rate,
            "Percent_income": amount / (income if income > 0 else 1e-6),
            "Cred_length": cred_length,
        }

        raw_input_df = pd.DataFrame([raw_input_data])
        processed_input_array = full_ml_pipeline.transform(raw_input_df)
        processed_input_df = pd.DataFrame(processed_input_array, columns=model_feature_names)

        y_pred_proba = best_lgb_model.predict_proba(processed_input_df)[:, 1]
        predicted_probability = float(y_pred_proba[0])

        risk_category = categorize_risk(predicted_probability)
        loan_decision = map_to_decision(risk_category)
        suggested_interest_rate = calculate_interest_rate(risk_category)

        if isinstance(lime_background_data, pd.DataFrame):
            background_array = lime_background_data.values
            valid_cols_mask = lime_background_data.std().values > 1e-6
        else:
            background_array = lime_background_data
            valid_cols_mask = np.std(lime_background_data, axis=0) > 1e-6

        filtered_background_data = background_array[:, valid_cols_mask]
        filtered_feature_names = [model_feature_names[i] for i, mask in enumerate(valid_cols_mask) if mask]
        filtered_input_values = processed_input_df.iloc[0].values[valid_cols_mask]

        explainer_lime = lime.lime_tabular.LimeTabularExplainer(
            training_data=filtered_background_data,
            feature_names=filtered_feature_names,
            class_names=["Non-Default", "Default"],
            mode="classification"
        )

        def predict_fn_wrapper(data):
            full_data = np.zeros((data.shape[0], len(model_feature_names)))
            full_data[:, valid_cols_mask] = data
            return best_lgb_model.predict_proba(full_data)

        exp = explainer_lime.explain_instance(
            data_row=filtered_input_values,
            predict_fn=predict_fn_wrapper,
            num_features=10
        )

        explanation_lines = []
        for feature, weight in exp.as_list():
            plain_english = get_plain_english_explanation(feature, weight, raw_input_data)
            explanation_lines.append(f"<li>{plain_english} (LIME weight: {weight:.4f})</li>")

        results_html = f"""
        <div style="padding:16px;border-radius:12px;border:1px solid #ddd;">
            <h3>Assessment Results</h3>
            <p><b>Predicted Probability of Default:</b> {predicted_probability:.4f}</p>
            <p><b>Risk Category:</b> {risk_category}</p>
            <p><b>Loan Decision:</b> {loan_decision}</p>
            <p><b>Suggested Interest Rate:</b> {suggested_interest_rate}</p>
            <hr>
            <h3>Explanation</h3>
            <p>The following factors contributed to this decision:</p>
            <ul>
                {''.join(explanation_lines)}
            </ul>
        </div>
        """

        return results_html

    except Exception as e:
        return f"<div style='color:red;'><b>Error:</b> {str(e)}</div>"


disclaimer_html = """
<div style="padding:16px;border-radius:12px;border:1px solid #f0c36d;background:#fff8e1;">
    <h3>Important Disclaimers and Considerations</h3>
    <ul>
        <li><b>This application is for demonstration and educational purposes only.</b></li>
        <li><b>Not for Real-World Decisions:</b> This model and its predictions should not be used for actual loan approval or rejection decisions in production.</li>
        <li><b>Fairness and Bias:</b> This demo does not include comprehensive fairness or bias auditing across demographic groups.</li>
        <li><b>Regulatory Compliance:</b> This model is not validated for regulatory compliance.</li>
        <li><b>Data Limitations:</b> Performance depends on the quality and representativeness of the training data.</li>
        <li><b>Continuous Monitoring:</b> Production models require monitoring, re-training, and periodic validation.</li>
        <li><b>Feature Scaling Note:</b> The model processes features internally, and the LIME explanations refer to processed features.</li>
    </ul>
</div>
"""

with gr.Blocks(title="Credit Risk Assessment") as demo:
    gr.Markdown("# 💰 Explainable AI Credit Risk Assessment")
    gr.Markdown("This application predicts the likelihood of loan default and provides explanations using an optimized LightGBM model.")

    with gr.Row():
        age = gr.Number(label="Age", value=30)
        income = gr.Number(label="Annual Income", value=50000)
        emp_length = gr.Number(label="Employment Length (years)", value=5.0)

    with gr.Row():
        amount = gr.Number(label="Loan Amount", value=10000)
        rate = gr.Number(label="Interest Rate (%)", value=10.0)
        cred_length = gr.Number(label="Credit History Length (years)", value=5)

    with gr.Row():
        home_ownership = gr.Dropdown(
            choices=["RENT", "MORTGAGE", "OWN", "OTHER"],
            value="RENT",
            label="Home Ownership"
        )
        loan_intent = gr.Dropdown(
            choices=["PERSONAL", "EDUCATION", "MEDICAL", "VENTURE", "HOMEIMPROVEMENT", "DEBTCONSOLIDATION"],
            value="PERSONAL",
            label="Loan Intent"
        )

    assess_btn = gr.Button("Assess Credit Risk")
    output = gr.HTML(label="Results")
    gr.HTML(disclaimer_html)

    assess_btn.click(
        fn=assess_credit_risk,
        inputs=[age, income, emp_length, amount, rate, cred_length, home_ownership, loan_intent],
        outputs=output
    )

demo.launch()

Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


Exception in thread Thread-7 (_do_normal_analytics_request):
Traceback (most recent call last):
  File "c:\Users\Harry\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\analytics.py", line 131, in get_local_ip_address
    ip_address = httpx.get(
                 ^^^^^^^^^^
  File "c:\Users\Harry\AppData\Local\Programs\Python\Python311\Lib\site-packages\httpx\_api.py", line 159, in get
    return request(
           ^^^^^^^^
  File "c:\Users\Harry\AppData\Local\Programs\Python\Python311\Lib\site-packages\httpx\_api.py", line 86, in request
    return client.request(
           ^^^^^^^^^^^^^^^
  File "c:\Users\Harry\AppData\Local\Programs\Python\Python311\Lib\site-packages\httpx\_client.py", line 600, in request
    return self.send(
           ^^^^^^^^^^
  File "c:\Users\Harry\AppData\Local\Programs\Python\Python311\Lib\site-packages\httpx\_client.py", line 620, in send
    response = self.send_handling_redirects(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Us